# Notebook 6 — Train, Tune, and Evaluate

This notebook trains models to predict whether a delivered order will be late.

Workflow:
1. Load the processed features and targets from Notebook 5
2. Train a simple baseline model
3. Evaluate models using validation data
4. Tune/select the final model
5. Evaluate the selected model on the test set once
6. Save the final model and results

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [3]:
feature_artifact_dir = Path("artifacts/features")

X_train = np.load(
    feature_artifact_dir / "X_train_processed.npy"
)

X_validation = np.load(
    feature_artifact_dir / "X_validation_processed.npy"
)

X_test = np.load(
    feature_artifact_dir / "X_test_processed.npy"
)

y_train = np.load(
    feature_artifact_dir / "y_train.npy"
)

y_validation = np.load(
    feature_artifact_dir / "y_validation.npy"
)

y_test = np.load(
    feature_artifact_dir / "y_test.npy"
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_validation.shape, y_validation.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (69608, 78) (69608,)
Validation: (14916, 78) (14916,)
Test: (14917, 78) (14917,)


In [5]:
print("Training target distribution:")
print(pd.Series(y_train).value_counts())

print("\nTraining target proportions:")
print(pd.Series(y_train).value_counts(normalize=True))

Training target distribution:
0    64129
1     5479
Name: count, dtype: int64

Training target proportions:
0    0.921288
1    0.078712
Name: proportion, dtype: float64


In [6]:
baseline_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_validation)
baseline_probabilities = baseline_model.predict_proba(X_validation)[:, 1]

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [7]:
baseline_results = {
    "accuracy": accuracy_score(
        y_validation,
        baseline_predictions
    ),
    "precision": precision_score(
        y_validation,
        baseline_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_validation,
        baseline_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_validation,
        baseline_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        baseline_probabilities
    )
}

print("Baseline validation results:")
for metric, value in baseline_results.items():
    print(f"{metric}: {value:.4f}")

Baseline validation results:
accuracy: 0.9213
precision: 0.0000
recall: 0.0000
f1: 0.0000
roc_auc: 0.5000


In [8]:
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [9]:
logistic_predictions = logistic_model.predict(X_validation)
logistic_probabilities = logistic_model.predict_proba(X_validation)[:, 1]

logistic_results = {
    "accuracy": accuracy_score(
        y_validation,
        logistic_predictions
    ),
    "precision": precision_score(
        y_validation,
        logistic_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_validation,
        logistic_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_validation,
        logistic_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        logistic_probabilities
    )
}

print("Logistic Regression validation results:")
for metric, value in logistic_results.items():
    print(f"{metric}: {value:.4f}")

Logistic Regression validation results:
accuracy: 0.6333
precision: 0.1294
recall: 0.6388
f1: 0.2152
roc_auc: 0.6959


In [10]:
from sklearn.ensemble import RandomForestClassifier

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [12]:
rf_predictions = rf_model.predict(X_validation)
rf_probabilities = rf_model.predict_proba(X_validation)[:, 1]

rf_results = {
    "accuracy": accuracy_score(y_validation, rf_predictions),
    "precision": precision_score(
        y_validation, rf_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_validation, rf_predictions, zero_division=0
    ),
    "f1": f1_score(
        y_validation, rf_predictions, zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation, rf_probabilities
    )
}

print("Random Forest validation results:")
for metric, value in rf_results.items():
    print(f"{metric}: {value:.4f}")

Random Forest validation results:
accuracy: 0.8636
precision: 0.2777
recall: 0.4574
f1: 0.3456
roc_auc: 0.7886


In [13]:
threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.05):
    threshold_predictions = (
        rf_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(
            y_validation,
            threshold_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            threshold_predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            threshold_predictions,
            zero_division=0
        )
    })

threshold_results_df = pd.DataFrame(threshold_results)

print(threshold_results_df)

    threshold  precision    recall        f1
0        0.20   0.095439  0.962521  0.173659
1        0.25   0.116819  0.912266  0.207117
2        0.30   0.147536  0.821124  0.250130
3        0.35   0.179398  0.706133  0.286109
4        0.40   0.213738  0.617547  0.317565
5        0.45   0.240531  0.524702  0.329853
6        0.50   0.277663  0.457411  0.345560
7        0.55   0.307054  0.378194  0.338931
8        0.60   0.359733  0.321124  0.339334
9        0.65   0.385694  0.234242  0.291468
10       0.70   0.429245  0.155026  0.227785
11       0.75   0.529680  0.098807  0.166547
12       0.80   0.620000  0.052811  0.097331


In [14]:
best_threshold_row = threshold_results_df.loc[
    threshold_results_df["f1"].idxmax()
]

best_threshold = best_threshold_row["threshold"]

print("Best threshold:", best_threshold)
print("\nBest validation results:")
print(best_threshold_row)

Best threshold: 0.5

Best validation results:
threshold    0.500000
precision    0.277663
recall       0.457411
f1           0.345560
Name: 6, dtype: float64


In [15]:
final_validation_predictions = (
    rf_probabilities >= best_threshold
).astype(int)

print("Final Random Forest Validation Results")
print("=" * 45)

print(f"Accuracy:  {accuracy_score(y_validation, final_validation_predictions):.4f}")
print(f"Precision: {precision_score(y_validation, final_validation_predictions, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_validation, final_validation_predictions, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_validation, final_validation_predictions, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_validation, rf_probabilities):.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        final_validation_predictions,
        target_names=["On-time", "Late"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, final_validation_predictions))

Final Random Forest Validation Results
Accuracy:  0.8636
Precision: 0.2777
Recall:    0.4574
F1 Score:  0.3456
ROC-AUC:   0.7886

Classification Report:
              precision    recall  f1-score   support

     On-time       0.95      0.90      0.92     13742
        Late       0.28      0.46      0.35      1174

    accuracy                           0.86     14916
   macro avg       0.61      0.68      0.63     14916
weighted avg       0.90      0.86      0.88     14916


Confusion Matrix:
[[12345  1397]
 [  637   537]]


In [16]:
# Final evaluation on the untouched test set

test_probabilities = rf_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

test_results = {
    "accuracy": accuracy_score(
        y_test,
        test_predictions
    ),
    "precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        test_probabilities
    )
}

print("Final Test Results")
print("=" * 45)

for metric, value in test_results.items():
    print(f"{metric.capitalize():10s}: {value:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["On-time", "Late"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

Final Test Results
Accuracy  : 0.8596
Precision : 0.2680
Recall    : 0.4532
F1        : 0.3368
Roc_auc   : 0.7803

Classification Report:
              precision    recall  f1-score   support

     On-time       0.95      0.89      0.92     13743
        Late       0.27      0.45      0.34      1174

    accuracy                           0.86     14917
   macro avg       0.61      0.67      0.63     14917
weighted avg       0.90      0.86      0.88     14917


Confusion Matrix:
[[12290  1453]
 [  642   532]]


In [17]:
from pathlib import Path
import json
import joblib

model_artifact_dir = Path("artifacts/model")
model_artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    rf_model,
    model_artifact_dir / "random_forest_model.joblib"
)

with open(
    model_artifact_dir / "decision_threshold.json",
    "w"
) as f:
    json.dump(
        {
            "threshold": float(best_threshold)
        },
        f,
        indent=2
    )

print("Model and threshold saved successfully.")

Model and threshold saved successfully.


In [18]:
results_summary = {
    "baseline": baseline_results,
    "logistic_regression_validation": logistic_results,
    "random_forest_validation": rf_results,
    "final_model": "RandomForestClassifier",
    "decision_threshold": float(best_threshold),
    "final_test_results": test_results
}

with open(
    model_artifact_dir / "results_summary.json",
    "w"
) as f:
    json.dump(
        results_summary,
        f,
        indent=2
    )

print("Results summary saved successfully.")

Results summary saved successfully.


In [19]:
summary_text = f"""
Notebook 6 — Train, Tune, and Evaluate
=======================================

Final model:
RandomForestClassifier

Decision threshold:
{best_threshold:.2f}

Validation performance:
Accuracy:  {rf_results["accuracy"]:.4f}
Precision: {rf_results["precision"]:.4f}
Recall:    {rf_results["recall"]:.4f}
F1:        {rf_results["f1"]:.4f}
ROC-AUC:   {rf_results["roc_auc"]:.4f}

Final test performance:
Accuracy:  {test_results["accuracy"]:.4f}
Precision: {test_results["precision"]:.4f}
Recall:    {test_results["recall"]:.4f}
F1:        {test_results["f1"]:.4f}
ROC-AUC:   {test_results["roc_auc"]:.4f}

The test set was used only once for the final evaluation.
"""

with open(
    model_artifact_dir / "results_summary.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(summary_text)

print(summary_text)


Notebook 6 — Train, Tune, and Evaluate

Final model:
RandomForestClassifier

Decision threshold:
0.50

Validation performance:
Accuracy:  0.8636
Precision: 0.2777
Recall:    0.4574
F1:        0.3456
ROC-AUC:   0.7886

Final test performance:
Accuracy:  0.8596
Precision: 0.2680
Recall:    0.4532
F1:        0.3368
ROC-AUC:   0.7803

The test set was used only once for the final evaluation.



In [20]:
print("Notebook 6 artifacts:")
for file in sorted(model_artifact_dir.iterdir()):
    print("-", file.name)

Notebook 6 artifacts:
- decision_threshold.json
- random_forest_model.joblib
- results_summary.json
- results_summary.txt


In [21]:
from pathlib import Path

print("=" * 60)
print("TASK 2 FINAL ARTIFACT VERIFICATION")
print("=" * 60)

# Notebooks
print("\nNotebooks:")
for notebook in sorted(Path(".").glob("*.ipynb")):
    print("-", notebook.name)

# Artifacts
print("\nArtifacts:")
for path in sorted(Path("artifacts").rglob("*")):
    if path.is_file():
        print("-", path)

print("\nVerification complete.")

TASK 2 FINAL ARTIFACT VERIFICATION

Notebooks:
- 01_read_and_join.ipynb
- 02_create_labels.ipynb
- 03_train_val_test_split.ipynb
- 04_eda.ipynb
- 05_feature_engineering.ipynb
- 06_train_tune_evaluate.ipynb

Artifacts:
- artifacts\eda\target_distribution.png
- artifacts\features\feature_manifest.json
- artifacts\features\feature_names.json
- artifacts\features\preprocessor.joblib
- artifacts\features\X_test_processed.npy
- artifacts\features\X_train_processed.npy
- artifacts\features\X_validation_processed.npy
- artifacts\features\y_test.npy
- artifacts\features\y_train.npy
- artifacts\features\y_validation.npy
- artifacts\labeled_orders.csv
- artifacts\model\decision_threshold.json
- artifacts\model\random_forest_model.joblib
- artifacts\model\results_summary.json
- artifacts\model\results_summary.txt
- artifacts\order_level_ml_table.csv
- artifacts\test.csv
- artifacts\train.csv
- artifacts\validation.csv

Verification complete.
